# Cubic budget persistence

This notebook is the slower companion to `reports/cubic-budget-persistence.md`.

The useful question is narrow: once the Newton cutoff rises, which parts of the old cubic drama cool away, and which parts keep showing up as real slow geometry?


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

rows = []
with (Path('..') / 'art' / 'cubic-budget-persistence.csv').open() as handle:
    for row in csv.DictReader(handle):
        parsed = {
            'polynomial': row['polynomial'],
            'budget': int(row['budget']),
            'tile_x': int(row['tile_x']),
            'tile_y': int(row['tile_y']),
            'sample_count': int(row['sample_count']),
            'mean_iterations': float(row['mean_iterations']),
            'late_fraction': float(row['late_fraction']),
            'unresolved_fraction': float(row['unresolved_fraction']),
        }
        rows.append(parsed)
rows[0]


In [ ]:
grouped = defaultdict(list)
for row in rows:
    grouped[(row['polynomial'], row['budget'])].append(row)

def summary(block):
    total = sum(row['sample_count'] for row in block)
    grid_late = sum(row['sample_count'] * row['late_fraction'] for row in block) / total
    unresolved = sum(row['sample_count'] * row['unresolved_fraction'] for row in block) / total
    center = sorted(block, key=lambda row: abs((row['tile_x'] + 0.5) - 6) + abs((row['tile_y'] + 0.5) - 6))[:4]
    center_late = sum(row['late_fraction'] for row in center) / len(center)
    return grid_late, center_late, unresolved

for key in sorted(grouped):
    grid_late, center_late, unresolved = summary(grouped[key])
    print(f'{key}: grid late={grid_late:.3f}, center={center_late:.3f}, unresolved={unresolved:.3f}')


## What to keep

If the high-budget maps still keep a hot center for the unity cubic while the asymmetric cubic cools, that is the real result.

The low-budget picture alone is not enough, because unresolved starts can fake extra drama. The higher-budget pass is the check that separates cutoff noise from a persistent slow core.
